**Building a Document-Based RAG (Retrieval-Augmented Generation) System using LangChain and Local Ollama Models**

**🔹 Step 1: Install System Dependencies and Ollama**

In [16]:
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh


Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
37 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InR

**🔹 Step 2: Start Ollama Server in Background**

In [17]:
import time
import subprocess
import threading

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()

time.sleep(15)


**🔹 Step 3: Pull Required Models from Ollama**

In [18]:
!ollama pull mistral
!ollama pull nomic-embed-text


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling ff82381e2bea: 100% ▕▏ 4.1 GB                         
pulling 43070e2d4e53: 100% ▕▏  11 KB                         
pulling 491dfa501e59: 100% ▕▏  801 B                         
pulling ed11eda7790d: 100% ▕▏   30 B                         
pulling 42347cd80dc8: 100% ▕▏  485 B                         
verifying sha256 digest 
writing manifest 
success 
pulling manifest ⠙ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠼ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠧ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠋ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠸ pulling manifest ⠸ pulling manifest 
pulling 970aa74c0a90: 100% ▕▏ 274 MB                         
pulling c71d239df917: 100% ▕▏  11 KB                         
pulling ce4a164fc046: 100% ▕▏   17 B                        

**🔹 Step 4: Install Python Libraries**

In [ ]:
!pip install langchain-ollama langchain-community langchain langchain-core unstructured unstructured[all-docs] ollama python-pptx PyPDF2 faiss-cpu


**🔹 Step 5: Load PDF and Convert to LangChain Documents**

In [20]:
from PyPDF2 import PdfReader
from langchain.docstore.document import Document

uploaded_file = "/content/budget_speech (1)-1.pdf"

document = []
reader = PdfReader(uploaded_file)
for i, page in enumerate(reader.pages, 1):
    document.append(Document(page_content=page.extract_text(), metadata={'page': i}))


**🔹 Step 6: Split Documents into Chunks**

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
data_chunks = text_splitter.split_documents(document)


**🔹 Step 7: Create Embeddings and FAISS Vector Store**

In [ ]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(data_chunks, OllamaEmbeddings(model="nomic-embed-text", show_progress=True))
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 5})


**🔹 Step 8: Initialize Local LLM from Ollama**

In [23]:
from langchain_community.chat_models import ChatOllama

local_model = "mistral"
llm = ChatOllama(model=local_model, num_predict=1000, stop=["<|start_header_id|>", "<|end_header_id|>", "<|eot_id|>"])


**🔹 Step 9: Define Prompt Template for RAG**

In [32]:
# Set up the RAG chain:
prompt_template = """
You are an AI assistant. Use the provided context to generate a precise and informative answer to the user’s question. If the context does not contain relevant information, indicate that you don’t have enough details to answer.

**Question:** {question}
**Context:** {context}

"""

In [33]:
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template,
)

**🔹 Step 10: Setup RAG Pipeline (Chain)**

In [34]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


**🔹 Step 11: Ask a Question and Get the Answer**

In [37]:
question = """provide me  10 import point of gst"""
print(question)
print(rag_chain.invoke(question))


provide me  10 import point of gst



OllamaEmbeddings: 100%|██████████| 1/1 [00:00<00:00, 50.82it/s]


 Based on the provided context, here are ten important points related to Goods and Services Tax (GST) and Indirect Taxes:

1. Amendment in section 2(61) of the CGST Act, 2017 to allow Input Service Distributor to distribute input tax credit for inter-state supplies on which tax has to be paid on reverse charge basis. This amendment will take effect from 1st April, 2025.

2. Amendment in section 2(69)(c) of the CGST Act, 2017 to provide definitions for 'Local Fund' and 'Municipal Fund' used in the definition of "local authority".

3. Services provided or agreed to be provided by insurance companies by way of reinsurance services under the Weather Based Crop Insurance Scheme (WBCIS) and the Modified National Agricultural Insurance Scheme (MNAIS) are exempted from service tax for the period commencing from 1st April, 2011 and ending with 30th June, 2017.

4. Customs duty rate changes aim to reduce input costs, deepen value addition, promote export competitiveness, correct inverted duty st